<a id="inicio"></a>

# Electricity Demand Monitoring with InfluxDB — Part 1: Data Ingestion

## Objective

Build the ingestion backbone of a real-time electricity demand monitoring system. Extract hourly demand from the Spanish grid operator's public API, process it with proper timezone handling, and continuously land it in a time-series database for dashboarding.

## Methodology

- **Source**: [Red Eléctrica de España (REE) REDData API](https://www.ree.es/es/apidatos) — hourly electricity demand data for peninsular Spain
- **Storage**: InfluxDB 2.x running locally via Docker, with a `capstone7` bucket
- **Ingestion pipeline**: Time-aware extraction with Madrid → UTC conversion (including daylight-saving transitions), hourly resampling, and daily batched writes
- **Idempotent backfill**: A `pickle`-persisted `set` of already-ingested days lets the ingestion loop restart cleanly without reprocessing
- **Dashboard**: InfluxDB Graph, Table, and Gauge panels built with Flux queries

## Why a time-series database

Traditional RDBMSs can store time-series data but struggle with the ingestion rate and the downsampling / retention policy patterns common in monitoring systems. InfluxDB is purpose-built for high-cardinality time-series and ships with its own query language (Flux) optimized for window aggregation.

<div class="alert alert-block alert-info">

### ℹ️ Note on cell outputs

The cell outputs visible in this notebook were captured during the **original run** and are shown here as a visual reference for portfolio review. Some artifacts may still appear in Spanish because the outputs predate the English code translation.

**To re-run locally:**

1. This notebook uses **InfluxDB 2.x** running in a Docker container. Launch the stack with:
   `docker compose up -d`
   (the `Dockerfile` and `docker-compose.yml` are in the repo root)
2. Install Python dependencies:
   `pip install -r requirements.txt`
3. The InfluxDB default token is `mcidaen_token` (set in `docker-compose.yml`). In production, swap this out for a generated token via InfluxDB's UI and load it from an environment variable.
4. The [Red Eléctrica de España public API](https://www.ree.es/es/apidatos) requires no authentication.
5. Launch Jupyter, open this file, and run top-to-bottom. The ingestion loop will fetch historical data into your InfluxDB instance.

</div>

---

---

<a id="indice"></a>
## Contents

* [1. Introduction](#section1)
* [2. Extracting Demand Data](#section2)
* [3. Ingestion and Visualization](#section3)

---

In [1]:
import matplotlib.pyplot as plt

# Optimiza los gráficos para pantalla retina
%config InlineBackend.figure_format = 'retina'
# Por defecto usamos el backend inline
%matplotlib inline

# Oculta warnings
import warnings
warnings.simplefilter('ignore')

# La libreta ocupa así el 95% de la pantalla
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))

---

<a id="section1"></a>
## 1. Introduction

This project builds a prototype system for visualizing and forecasting Spanish hourly electricity demand. The prototype includes:

- **Electricity demand extraction from Red Eléctrica** — pull peninsular demand from REE's public data API
- **Continuous InfluxDB ingestion** — land the extracted data into InfluxDB so we can visualize both the raw series and derived metrics
- **Day-ahead demand forecasting model** — train a forecasting model from the ingested history
- **Forecast ingestion** — write the forecast back into the database so it's viewable alongside the actual demand

This notebook is split into two parts to cleanly separate ingestion from forecasting. You'll want to make meaningful progress on Part 1 before attempting Part 2 — the forecaster needs ingested history to train on.

Below are the imports we'll need across the notebook. Add anything else you want.

In [2]:
import requests
import json
import pandas as pd
import datetime
import uuid
import pickle
import time

from influxdb_client import InfluxDBClient, Point, WritePrecision
from influxdb_client.client.write_api import SYNCHRONOUS

---

<a id="section2"></a>
## 2. Extracting Demand Data

Spanish peninsular electricity demand is available via [REDData](https://www.ree.es/es/apidatos), REE's free data portal. The portal exposes a broad catalog — generation, demand, market balance, etc. — and for this project we use the demand subset.

REE ships a [live widget](https://demanda.ree.es/visiona/peninsula/nacionalau/total) with current demand (MW) across different timeframes. The widget internally calls a handful of APIs for both generation and demand; we'll use two of them.

First, the real-demand API:

In [3]:
import requests
import json

r = requests.get('https://demanda.ree.es/WSvisionaMovilesPeninsulaRest/resources/demandaGeneracionPeninsula?callback=&curva=NACIONALAU&fecha=2025-03-22')

response = r.text.replace('(', '').replace(')', '').replace(';', '')

data_dem = json.loads(response)['valoresHorariosGeneracion']
data_dem


[{'ts': '2025-03-21 21:00',
  'dem': 35475.0,
  'eol': 14430.0,
  'nuc': 5673.0,
  'car': 389.0,
  'cc': 4087.0,
  'hid': 9082.0,
  'inter': -595.0,
  'sol': 101.0,
  'solFot': 40.0,
  'solTer': 61.0,
  'termRenov': 362.0,
  'die': 341.0,
  'gas': 105.0,
  'vap': 189.0,
  'genAux': 0.0,
  'cogenResto': 1096.0,
  'expAnd': -55,
  'expMar': -390,
  'expPor': 0,
  'expFra': -2135,
  'expTot': -2580,
  'impFra': 0,
  'impPor': 2078,
  'impMar': 0,
  'impAnd': 0,
  'impTot': 2078,
  'conb': -350.0,
  'turb': 1105.0,
  'gnhd': 7715.0,
  'bat': 7,
  'consBat': 0,
  'dif': 7},
 {'ts': '2025-03-21 21:05',
  'dem': 35437.0,
  'eol': 14347.0,
  'nuc': 5673.0,
  'car': 386.0,
  'cc': 3890.0,
  'hid': 9149.0,
  'inter': -412.0,
  'sol': 101.0,
  'solFot': 40.0,
  'solTer': 61.0,
  'termRenov': 363.0,
  'die': 338.0,
  'gas': 105.0,
  'vap': 189.0,
  'genAux': 0.0,
  'cogenResto': 1096.0,
  'expAnd': -54,
  'expMar': -402,
  'expPor': 0,
  'expFra': -2301,
  'expTot': -2757,
  'impFra': 0,
  'impPor

The response is a list of JSON objects, one per timestamp. The field of interest is `dem` (demand in MW). Intervals are not perfectly regular.

Second, the forecast-demand API:

In [4]:

import requests
import json

r = requests.get('https://demanda.ree.es/WSvisionaMovilesPeninsulaRest/resources/prevProgPeninsula?callback=&curva=NACIONALAU&fecha=2025-03-22')

response = r.text.replace('(', '').replace(')', '').replace(';', '')

data_prev = json.loads(response)['valoresPrevistaProgramada']
data_prev


[{'ts': '2025-03-21 21:00', 'pro': 35569, 'pre': 35352},
 {'ts': '2025-03-21 21:05', 'pro': 35569, 'pre': 35607},
 {'ts': '2025-03-21 21:10', 'pro': 35569, 'pre': 35661},
 {'ts': '2025-03-21 21:15', 'pro': 35369, 'pre': 35513},
 {'ts': '2025-03-21 21:20', 'pro': 35369, 'pre': 35271},
 {'ts': '2025-03-21 21:25', 'pro': 35369, 'pre': 35043},
 {'ts': '2025-03-21 21:30', 'pro': 34767, 'pre': 34827},
 {'ts': '2025-03-21 21:35', 'pro': 34767, 'pre': 34604},
 {'ts': '2025-03-21 21:40', 'pro': 34767, 'pre': 34354},
 {'ts': '2025-03-21 21:45', 'pro': 33725, 'pre': 34076},
 {'ts': '2025-03-21 21:50', 'pro': 33725, 'pre': 33774},
 {'ts': '2025-03-21 21:55', 'pro': 33725, 'pre': 33449},
 {'ts': '2025-03-21 22:00', 'pro': 32689, 'pre': 33080},
 {'ts': '2025-03-21 22:05', 'pro': 32689, 'pre': 32737},
 {'ts': '2025-03-21 22:10', 'pro': 32689, 'pre': 32421},
 {'ts': '2025-03-21 22:15', 'pro': 31766, 'pre': 32134},
 {'ts': '2025-03-21 22:20', 'pro': 31766, 'pre': 31868},
 {'ts': '2025-03-21 22:25', 'pr

This response has the same shape, with field `pre` (prediction in MW) per (variable-interval) timestamp.

We care about those two fields — that way we can compare our own forecast against REE's own prediction. Modeling this as a time series in InfluxDB is straightforward:

- **Measurement**: `demand`
- **Tags**: none (no additional dimensions needed)
- **Fields**: `RealDemand` (measured) and `ForecastREE` (REE's own forecast)

Before we can write to InfluxDB we need helpers to parse the API responses and to ingest the resulting DataFrames.

---

### Task 1: Parse API responses into a DataFrame

Implement a function that takes the two API responses and returns a DataFrame with:

- Two columns: `RealDemand` (from the real-demand API) and `ForecastREE` (from the forecast API)
- A **UTC** datetime index. The API returns values in Madrid local time, so you'll need to handle the timezone conversion — including daylight-saving transitions (Madrid shifts between CET and CEST twice a year)
- Hourly frequency (aggregate with mean)

Don't worry about fetching the data in this step — just the transformation.

*Hint*: The time-series preparation exercises from earlier in the module are directly applicable here.

In [5]:
import pandas as pd
import pytz

def process_response(data_dem, data_prev):
    """
    Convierte los datos de demanda real y forecast de REE en un DataFrame unificado.
    
    - data_dem: lista de diccionarios con demanda real ('ts' y 'dem')
    - data_prev: lista de diccionarios con forecast ('ts' y 'pro')
    
    Devuelve un DataFrame con índice temporal en UTC y frecuencia horaria,
    con columnas RealDemand y ForecastREE.
    """
    
    # Convertir listas a DataFrames
    df_dem = pd.DataFrame(data_dem)[['ts', 'dem']].rename(columns={'ts': 'timestamp', 'dem': 'RealDemand'})
    df_prev = pd.DataFrame(data_prev)[['ts', 'pro']].rename(columns={'ts': 'timestamp', 'pro': 'ForecastREE'})
    
    # Unir los DataFrames por timestamp
    df = pd.merge(df_dem, df_prev, on='timestamp', how='outer')
    
    # Convertir timestamp a datetime con zona horaria Madrid
    madrid = pytz.timezone('Europe/Madrid')
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['timestamp'] = df['timestamp'].dt.tz_localize(madrid, ambiguous='NaT', nonexistent='shift_forward')
    
    # Convertir a UTC
    df['timestamp'] = df['timestamp'].dt.tz_convert('UTC')
    
    # Establecer como índice
    df.set_index('timestamp', inplace=True)
    
    # Reagrupar a frecuencia horaria con media
    df = df.resample('1H').mean()
    
    return df



In [6]:
result = process_response(data_dem, data_prev)
result

,RealDemand,ForecastREE
timestamp,,
2025-03-21 20:00:00+00:00,34851.750000,34857.50
2025-03-21 21:00:00+00:00,31784.500000,31475.50
2025-03-21 22:00:00+00:00,28943.250000,28891.25
2025-03-21 23:00:00+00:00,26998.916667,26665.25
2025-03-22 00:00:00+00:00,25154.083333,24923.00
2025-03-22 01:00:00+00:00,23918.666667,23546.25
2025-03-22 02:00:00+00:00,23052.916667,22851.75
2025-03-22 03:00:00+00:00,22541.916667,22370.00
2025-03-22 04:00:00+00:00,22532.000000,22322.50


Expected output (first 32 rows):

|                           |   RealDemand |   ForecastREE |
|:--------------------------|-------------:|--------------:|
| 2025-03-21 20:00:00+00:00 |      34851.8 |       34794.2 |
| 2025-03-21 21:00:00+00:00 |      31784.5 |       31565.4 |
| 2025-03-21 22:00:00+00:00 |      28943.2 |       29231.0 |
| 2025-03-21 23:00:00+00:00 |      26998.9 |       26842.3 |
| 2025-03-22 00:00:00+00:00 |      25154.1 |       25041.5 |
| 2025-03-22 01:00:00+00:00 |      23918.7 |       23511.2 |
| 2025-03-22 02:00:00+00:00 |      23052.9 |       22898.8 |
| 2025-03-22 03:00:00+00:00 |      22541.9 |       22352.7 |
| 2025-03-22 04:00:00+00:00 |      22532.0 |       22171.3 |
| 2025-03-22 05:00:00+00:00 |      23294.0 |       23121.2 |
| 2025-03-22 06:00:00+00:00 |      24389.1 |       24771.9 |
| 2025-03-22 07:00:00+00:00 |      25771.0 |       25225.7 |
| 2025-03-22 08:00:00+00:00 |      27635.4 |       27188.1 |
| 2025-03-22 09:00:00+00:00 |      28277.2 |       27981.8 |
| 2025-03-22 10:00:00+00:00 |      28207.3 |       29118.2 |
| 2025-03-22 11:00:00+00:00 |      28266.0 |       27718.8 |
| 2025-03-22 12:00:00+00:00 |      28530.7 |       28758.0 |
| 2025-03-22 13:00:00+00:00 |      28173.4 |       28535.6 |
| 2025-03-22 14:00:00+00:00 |      27078.4 |       27267.6 |
| 2025-03-22 15:00:00+00:00 |      26660.8 |       26690.2 |
| 2025-03-22 16:00:00+00:00 |      27372.6 |       26619.3 |
| 2025-03-22 17:00:00+00:00 |      28190.0 |       28014.3 |
| 2025-03-22 18:00:00+00:00 |      30235.0 |       30348.3 |
| 2025-03-22 19:00:00+00:00 |      31836.1 |       31231.2 |
| 2025-03-22 20:00:00+00:00 |      31600.1 |       32256.9 |

The end goal is continuous ingestion into InfluxDB. The DataFrame format above is useful for both analysis and ingestion — InfluxDB's Python client accepts DataFrames directly. Next step: helper that writes our DataFrame into the `capstone7` bucket.

---

### Task 2: Write a DataFrame to InfluxDB

Implement a function that ingests a DataFrame like the one above into InfluxDB. Use a bucket named `capstone7` (create it first through the InfluxDB UI).

With the client configured, write the DataFrame. Measurement: `demand`. Fields: `RealDemand` (actual) and `ForecastDemand` (REE's forecast).

After the helper is in place, run the ingestion code and capture a screenshot of the data in InfluxDB's Data Explorer to verify the timestamps landed correctly.

*Hint*: The module's InfluxDB ingestion exercise covers the mechanics.

In [7]:
# Podéis incluir la información de vuestra instancia de InfluxDB a continuación
token = "mcidaen_token"
org = "mcidaen"
bucket = "capstone7"

client = InfluxDBClient(url="http://influxdb:8086", token=token)

In [8]:
try:
    buckets = client.buckets_api().find_buckets().buckets
    print("✅ Conexión exitosa. Buckets disponibles:")
    for b in buckets:
        print(f"- {b.name}")
except Exception as e:
    print("❌ Error al conectar con InfluxDB:", e)

✅ Conexión exitosa. Buckets disponibles:
- _tasks
- mod7
- _monitoring
- capstone7


In [40]:
def save_to_influxdb(df: pd.DataFrame, client: InfluxDBClient, bucket: str, org: str) -> pd.DataFrame:
    """
    Escribe el DataFrame en InfluxDB como medida 'demand' con campos RealDemand y ForecastDemand.
    """
    write_api = client.write_api(write_options=SYNCHRONOUS)
    
    # Recorrer las filas del DataFrame e insertar puntos en InfluxDB
    for index, row in df.iterrows():
        point = (
            Point("demand")
            .field("RealDemand", float(row["RealDemand"]))
            .field("ForecastDemand", float(row["ForecastREE"]))
            .time(index, WritePrecision.NS)  # usar el índice como timestamp
        )
        write_api.write(bucket=bucket, org=org, record=point)
    
    print("✅ Datos guardados correctamente en InfluxDB")
    return df

In [10]:
save_to_influxdb(result, client, bucket, org)

✅ Datos guardados correctamente en InfluxDB


,RealDemand,ForecastREE
timestamp,,
2025-03-21 20:00:00+00:00,34851.750000,34857.50
2025-03-21 21:00:00+00:00,31784.500000,31475.50
2025-03-21 22:00:00+00:00,28943.250000,28891.25
2025-03-21 23:00:00+00:00,26998.916667,26665.25
2025-03-22 00:00:00+00:00,25154.083333,24923.00
2025-03-22 01:00:00+00:00,23918.666667,23546.25
2025-03-22 02:00:00+00:00,23052.916667,22851.75
2025-03-22 03:00:00+00:00,22541.916667,22370.00
2025-03-22 04:00:00+00:00,22532.000000,22322.50


### Verification

Open the Data Explorer and confirm you see a demand curve with correct timestamps:

![tarea2](img/tarea2.png)

My result:

![tarea2](img_luis/tarea_2.png)

At this point we can parse API responses and ingest them into InfluxDB. Next, we wrap the API call itself so we can pull historical data programmatically.

---

### Task 3: API fetch function

Implement a function that pulls real-demand and forecast-demand data for a specific date (`dt: date`) and returns two JSON objects, which we then parse with the function from Task 1.

*Optional*: To guard against transient API failures, implement retry logic — sleep between attempts and catch `requests.RequestException` or its [subclasses](https://requests.readthedocs.io/en/master/_modules/requests/exceptions/).

In [55]:
def get_demand_data(dt: datetime.date) -> dict:
    """
    Obtiene los datos de demanda y previsión para una fecha específica
    """
    import requests
    import json
    import time
    from requests.exceptions import RequestException

    fecha = dt.strftime('%Y-%m-%d')
    
    url_dem = f"https://demanda.ree.es/WSvisionaMovilesPeninsulaRest/resources/demandaGeneracionPeninsula?callback=&curva=NACIONALAU&fecha={fecha}"
    url_prev = f"https://demanda.ree.es/WSvisionaMovilesPeninsulaRest/resources/prevProgPeninsula?callback=&curva=NACIONALAU&fecha={fecha}"

    # Reintento simple
    for intento in range(3):
        try:
            r_dem = requests.get(url_dem)
            r_prev = requests.get(url_prev)

            text_dem = r_dem.text.replace("(", "").replace(")", "").replace(";", "")
            text_prev = r_prev.text.replace("(", "").replace(")", "").replace(";", "")

            dem = json.loads(text_dem)["valoresHorariosGeneracion"]
            prev = json.loads(text_prev)["valoresPrevistaProgramada"]
            
            return dem, prev

        except RequestException as e:
            print(f"Error al conectar (intento {intento+1}/3):", e)
            time.sleep(2)
        except json.JSONDecodeError as e:
            print("Error al decodificar JSON:", e)
            break

    raise ValueError("No se pudieron obtener los datos tras varios intentos.")


In [ ]:
dem, prev = get_demand_data(
    datetime.date(2025, 3, 18)
)

In [56]:
process_response(dem, prev)

,RealDemand,ForecastREE
timestamp,,
2025-03-17 20:00:00+00:00,37474.916667,37461.75
2025-03-17 21:00:00+00:00,33838.166667,33886.50
2025-03-17 22:00:00+00:00,30280.583333,30081.50
2025-03-17 23:00:00+00:00,27658.666667,27771.00
2025-03-18 00:00:00+00:00,25658.916667,25693.50
2025-03-18 01:00:00+00:00,24539.083333,24376.00
2025-03-18 02:00:00+00:00,24001.166667,23900.00
2025-03-18 03:00:00+00:00,23916.833333,23921.50
2025-03-18 04:00:00+00:00,24595.083333,24877.25


Expected output for a March 17 2025 fetch:

|                           |   RealDemand |   ForecastREE |
|:--------------------------|-------------:|--------------:|
| 2025-03-17 20:00:00+00:00 |      37474.9 |       37484.2 |
| 2025-03-17 21:00:00+00:00 |      33838.2 |       34224.8 |
| 2025-03-17 22:00:00+00:00 |      30280.6 |       30691.2 |
| 2025-03-17 23:00:00+00:00 |      27658.7 |       28055.9 |
| 2025-03-18 00:00:00+00:00 |      25658.9 |       26210.2 |
| 2025-03-18 06:00:00+00:00 |      31585.1 |       31418.9 |
| 2025-03-18 12:00:00+00:00 |      34851.6 |       34070.2 |
| 2025-03-18 18:00:00+00:00 |      36378.5 |       36481.6 |
| 2025-03-19 00:00:00+00:00 |      26256.5 |       25364.5 |
| ...

With these three helpers in place we have everything we need for end-to-end extraction. Next section wires up the full continuous-ingestion loop.

---

<a id="section3"></a>
## 3. Ingestion and Visualization

For a reasonably robust ingestion process that tolerates intermittent failures, we maintain a set of already-ingested days and only fetch missing ones.

Technically, because InfluxDB upserts based on `(measurement, tag-set, timestamp)`, re-ingesting the same day would just overwrite with identical values. But since we're downloading a lot of data, we'd rather not repeat work. A persisted set of ingested days gives us idempotent, resumable ingestion.

The ingestion helper takes `begin_ts` (start date) and `ingested` (set of already-processed days) as input, walks day-by-day up to today, and returns the list of `datetime`s it actually processed. It issues one daily API call (REE's API has a per-request point limit). Feel free to extend or refactor it.

In [31]:
import pytz
from datetime import datetime, timedelta

def ingest_data(begin_ts: datetime, ingested: set) -> list:
    """
    Obtiene los datos de demanda desde begin_ts
    """

    def parse_timestamp(ts_str):
        tz = pytz.timezone("Europe/Madrid")
        if "2A:" in ts_str:
            ts_str = ts_str.replace("2A:", "02:")
            dt_naive = datetime.strptime(ts_str, "%Y-%m-%d %H:%M")
            dt_localized = tz.localize(dt_naive, is_dst=True)
        elif "2B:" in ts_str:
            ts_str = ts_str.replace("2B:", "02:")
            dt_naive = datetime.strptime(ts_str, "%Y-%m-%d %H:%M")
            dt_localized = tz.localize(dt_naive, is_dst=False)
        else:
            dt_naive = datetime.strptime(ts_str, "%Y-%m-%d %H:%M")
            try:
                dt_localized = tz.localize(dt_naive, is_dst=None)
            except pytz.AmbiguousTimeError:
                dt_localized = tz.localize(dt_naive, is_dst=False)
        return dt_localized.astimezone(pytz.UTC)

    def process_response(data_dem, data_prev):
        df_dem = pd.DataFrame(data_dem)[['ts', 'dem']].rename(columns={'ts': 'timestamp', 'dem': 'RealDemand'})
        df_prev = pd.DataFrame(data_prev)[['ts', 'pro']].rename(columns={'ts': 'timestamp', 'pro': 'ForecastREE'})
        df = pd.merge(df_dem, df_prev, on='timestamp', how='inner')
        df['timestamp'] = df['timestamp'].apply(parse_timestamp)
        df = df.set_index('timestamp')
        df = df.resample('H').mean().dropna()
        return df

    now = datetime.now()
    end_ts = datetime(now.year, now.month, now.day, now.hour, 0) - timedelta(seconds=1)
    begin = begin_ts
    end = min(begin + timedelta(days=1, minutes=-1), end_ts)
    l = []
    try:
        while begin < end_ts:
            if begin.date() not in ingested:
                print(f"Getting data for: {begin.date()}")
                data_dem, data_prev = get_demand_data(begin.date())
                if data_dem and data_prev:
                    df = process_response(data_dem, data_prev)
                    save_to_influxdb(df, client, bucket, org)
                    print("✅ Datos guardados correctamente en InfluxDB")
                    l.append(begin.date())
            begin += timedelta(days=1)
            end = min(begin + timedelta(days=1, minutes=-1), end_ts)
    except Exception as e:
        print(f"An exception occurred: {e}")
        return l
    return l


In [21]:
ingest_data(datetime.datetime(2025, 1, 1), set([]))

Getting data for: 2025-01-01
✅ Datos guardados correctamente en InfluxDB
Getting data for: 2025-01-02
✅ Datos guardados correctamente en InfluxDB
Getting data for: 2025-01-03
✅ Datos guardados correctamente en InfluxDB
Getting data for: 2025-01-04
✅ Datos guardados correctamente en InfluxDB
Getting data for: 2025-01-05
✅ Datos guardados correctamente en InfluxDB
Getting data for: 2025-01-06
✅ Datos guardados correctamente en InfluxDB
Getting data for: 2025-01-07
✅ Datos guardados correctamente en InfluxDB
Getting data for: 2025-01-08
✅ Datos guardados correctamente en InfluxDB
Getting data for: 2025-01-09
✅ Datos guardados correctamente en InfluxDB
Getting data for: 2025-01-10
✅ Datos guardados correctamente en InfluxDB
Getting data for: 2025-01-11
✅ Datos guardados correctamente en InfluxDB
Getting data for: 2025-01-12
✅ Datos guardados correctamente en InfluxDB
Getting data for: 2025-01-13
✅ Datos guardados correctamente en InfluxDB
Getting data for: 2025-01-14
✅ Datos guardados corr

[datetime.date(2025, 1, 1),
 datetime.date(2025, 1, 2),
 datetime.date(2025, 1, 3),
 datetime.date(2025, 1, 4),
 datetime.date(2025, 1, 5),
 datetime.date(2025, 1, 6),
 datetime.date(2025, 1, 7),
 datetime.date(2025, 1, 8),
 datetime.date(2025, 1, 9),
 datetime.date(2025, 1, 10),
 datetime.date(2025, 1, 11),
 datetime.date(2025, 1, 12),
 datetime.date(2025, 1, 13),
 datetime.date(2025, 1, 14),
 datetime.date(2025, 1, 15),
 datetime.date(2025, 1, 16),
 datetime.date(2025, 1, 17),
 datetime.date(2025, 1, 18),
 datetime.date(2025, 1, 19),
 datetime.date(2025, 1, 20),
 datetime.date(2025, 1, 21),
 datetime.date(2025, 1, 22),
 datetime.date(2025, 1, 23),
 datetime.date(2025, 1, 24),
 datetime.date(2025, 1, 25),
 datetime.date(2025, 1, 26),
 datetime.date(2025, 1, 27),
 datetime.date(2025, 1, 28),
 datetime.date(2025, 1, 29),
 datetime.date(2025, 1, 30),
 datetime.date(2025, 1, 31),
 datetime.date(2025, 2, 1),
 datetime.date(2025, 2, 2),
 datetime.date(2025, 2, 3),
 datetime.date(2025, 2, 4)

In [45]:

with open('ingested.db', 'rb') as f:
    ingested = pickle.load(f)

# Mostrar el último día ingestado
print("Último día ingestado:", max(ingested))

Último día ingestado: 2025-05-20


In [48]:
ingested

{datetime.date(2024, 1, 1),
 datetime.date(2024, 1, 2),
 datetime.date(2024, 1, 3),
 datetime.date(2024, 1, 4),
 datetime.date(2024, 1, 5),
 datetime.date(2024, 1, 6),
 datetime.date(2024, 1, 7),
 datetime.date(2024, 1, 8),
 datetime.date(2024, 1, 9),
 datetime.date(2024, 1, 10),
 datetime.date(2024, 1, 11),
 datetime.date(2024, 1, 12),
 datetime.date(2024, 1, 13),
 datetime.date(2024, 1, 14),
 datetime.date(2024, 1, 15),
 datetime.date(2024, 1, 16),
 datetime.date(2024, 1, 17),
 datetime.date(2024, 1, 18),
 datetime.date(2024, 1, 19),
 datetime.date(2024, 1, 20),
 datetime.date(2024, 1, 21),
 datetime.date(2024, 1, 22),
 datetime.date(2024, 1, 23),
 datetime.date(2024, 1, 24),
 datetime.date(2024, 1, 25),
 datetime.date(2024, 1, 26),
 datetime.date(2024, 1, 27),
 datetime.date(2024, 1, 28),
 datetime.date(2024, 1, 29),
 datetime.date(2024, 1, 30),
 datetime.date(2024, 1, 31),
 datetime.date(2024, 2, 1),
 datetime.date(2024, 2, 2),
 datetime.date(2024, 2, 3),
 datetime.date(2024, 2, 4)

This gives us a process we can run continuously to keep the dataset up to date. We persist the set of ingested days to a `pickle` file — everything except the current day (which is still accumulating).

First-run initialization: create an empty set and persist it to `ingested.db`. After the first run, comment this cell out so a re-run doesn't wipe your progress.

In [24]:
# import pickle
# ingested_file = 'ingested.db'
# ingested = set([])
# with open(ingested_file, 'wb') as f:
#     pickle.dump(ingested, f)

To keep the data fresh, rerun the following cell periodically. You can also change the start date if you want to bulk-backfill.

In [36]:
begin_ts = datetime(2024, 10, 25, 0, 0)

In [44]:
while True:
    with open(ingested_file, 'rb') as f:
        ingested = pickle.load(f)
    l = ingest_data(begin_ts, ingested)[:-1] # Remove last day to keep updating it
    ingested = ingested.union(set(l))
    with open(ingested_file, 'wb') as f:
        pickle.dump(ingested, f)
    print("Ingestion finished, sleeping for 10 seconds")
    time.sleep(60)

Getting data for: 2025-05-21
✅ Datos guardados correctamente en InfluxDB
✅ Datos guardados correctamente en InfluxDB
Ingestion finished, sleeping for 10 seconds


This process takes a while depending on the date range. While it runs, you can verify incoming data in InfluxDB and start Task 4 — or even jump ahead to Part 2.

### Task 4: Dashboard

Using the ingested data, build an InfluxDB dashboard with three panels:

- **Graph** panel plotting both `RealDemand` and `ForecastDemand`
- **Table** panel with three columns — `time`, `RealDemand`, `ForecastDemand`. This requires Flux's `pivot()` operator
- **Gauge** panel showing the difference between `RealDemand` and `ForecastDemand`

All panels should respect the dashboard's time-range filter.

Capture a screenshot of the finished dashboard and include the three Flux queries.

Expected result:

![tarea4](img/tarea4.png)

My result:

![tarea4](img_luis/tarea_41.png)

### Query 1 — Graph panel

```flux
from(bucket: "capstone7")
  |> range(start: v.timeRangeStart, stop: v.timeRangeStop)
  |> filter(fn: (r) => r["_measurement"] == "demand")
  |> filter(fn: (r) => r["_field"] == "ForecastDemand" or r["_field"] == "RealDemand")
  |> aggregateWindow(every: v.windowPeriod, fn: mean, createEmpty: false)
  |> yield(name: "mean")
```

### Query 2 — Gauge panel

```flux
from(bucket: "capstone7")
  |> range(start: v.timeRangeStart, stop: v.timeRangeStop)
  |> filter(fn: (r) => r["_measurement"] == "demand")
  |> filter(fn: (r) => r["_field"] == "ForecastDemand" or r["_field"] == "RealDemand")
  |> aggregateWindow(every: v.windowPeriod, fn: mean, createEmpty: false)
  |> yield(name: "mean")
```

### Query 3 — Table panel

```flux
from(bucket: "capstone7")
  |> range(start: v.timeRangeStart, stop: v.timeRangeStop)
  |> filter(fn: (r) => r["_measurement"] == "demand")
  |> filter(fn: (r) => r["_field"] == "ForecastDemand" or r["_field"] == "RealDemand")
  |> aggregateWindow(every: v.windowPeriod, fn: mean, createEmpty: false)
  |> yield(name: "mean")
```

We now have a working prototype of continuous electricity-demand ingestion. As long as the backfill loop keeps running the dataset stays current. If you stop it, the next run will backfill the gap to present. Safe to pause and resume anytime.

---